# M7 on Kaggle: full-pool DA-FedDistill (E3) and FedMD

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*. M7 fits 15 clients per window on CPU.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed. An interactive session can stop when it is left alone.
Time is an estimate: the E3 pass and the FedMD pass take about 2 hours together, so allow 3.

Each pass runs in its own process, and the zip is rewritten after each one, so a failure in FedMD does not lose E3. Every cell prints an exit code: 137 means the process was killed (usually out of memory), 139 a native crash. The log lines carry memory use.

**Output:** `/kaggle/working/m7_full_outputs.zip` contains `reports/m7/full/` (with any checkpoints) and the run folders. Download it and bring it back, also after a failure.


In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full


In [ ]:
import glob, os, shutil
def pack():
    stage = "/kaggle/working/m7_stage"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree("reports/m7/full", f"{stage}/reports/m7/full")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m7_distill_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive("/kaggle/working/m7_full_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir("reports/m7/full")))


In [ ]:
# E3 pass. 4 threads, matching the dev run and M6, because results reproduce exactly only at the same thread count
!python -X faulthandler scripts/m7_distill.py --env kaggle --pool full --set resources.n_jobs=4 --only e3 --resume; echo "exit code: $?"
pack()


In [ ]:
# FedMD pass
!python -X faulthandler scripts/m7_distill.py --env kaggle --pool full --set resources.n_jobs=4 --only fedmd --resume; echo "exit code: $?"
pack()


In [ ]:
# Report from whatever finished
!python -X faulthandler scripts/m7_distill.py --env kaggle --pool full --set resources.n_jobs=4 --report; echo "exit code: $?"
pack()
if os.path.exists("reports/m7/full/DISTILL.md"):
    print(open("reports/m7/full/DISTILL.md", encoding="utf-8").read()[:6000])
